# Generate Training Datasets (.npz)

This notebook orchestrates the conversion of DPX-server data (NAKO, TotalSegmentator, MSD) into the standardized `.npz` format used by the `DataLoader_npz` and `DataGenerator` pipeline.

> [!IMPORTANT]
> **SERVER ONLY**: This notebook and the imported converter scripts rely on the DPX/patchwork environment. They will **not** work on a local machine without access to `/software` and the appropriate environment variables (`DPXROOT`, `DPXproject`).

## Workflow
1. **Load**: Connect to DPX and pull raw volumes via `patchwork`.
2. **Resample**: Perform isotropic resampling to 1 mm³ voxel size.
3. **Tag**: Store modality information (CT vs MRI) for universal normalization.
4. **Save**: Export to compressed `.npz` archives.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in path
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 1. NAKO (GPUnet) Conversion
Converts NAKO MRI data (Body and Head).
- 11 brain images (each T1_crop + T1km_crop + T2_crop) = 33
- 7 body images (each 4 contrasts) = 28

In [ ]:
from data.train_data.Nako_to_npz import Nako_to_npz

# Body Training Data
loader_body = Nako_to_npz(mode='training_body')
loader_body.to_npz('../../data/test_data/GPUnet_body')

# Head Training Data (Optional)
# loader_head = Nako_to_npz(mode='training_head')
# loader_head.to_npz('../../data/test_data/GPUnet_head')

## 2. TotalSegmentator Conversion
Converts TotalSegmentator CT data. <br>
- 45 images in total

In [ ]:
from data.train_data.TotalSeg_to_npz import TotalSeg_to_npz

loader_ts = TotalSeg_to_npz(mode='train_combined')
loader_ts.to_npz('../../data/test_data/TotalSeg')

## 3. Medical Segmentation Decathlon (MSD) Conversion
Converts both CT tasks (Liver, Lung, Pancreas, Spleen) and MRI tasks (Brain, Prostate) into a single archive. <br>
- 4 images each = 24

In [ ]:
from data.train_data.MedDec_to_npz import MedDec_to_npz

loader_msd = MedDec_to_npz()
loader_msd.to_npz('../../data/test_data/MSD')

## Summary
After running the cells above, you should have the following files in `data/test_data/`:
- `GPUnet_body.npz`
- `TotalSeg.npz`
- `MSD.npz`

These can then be used in the `DataGenerator` by passing them to `DataLoader_npz`.